# Final Results and Claims

**RESULTS notebook — the locked configuration, dev and eval results, robustness, independent metrics, and the claim ledger**

This is the **results** notebook: it reports the locked configuration and the claim ledger.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## Locked configuration

Chosen on dev before the eval split was touched:

- **Scope:** hierarchical backoff (intersection → team → class → global), minimum evidence 2.
- **Prior:** empirical-Bayes centered Laplace, κ=2, cap ±0.20.
- **Scaling:** pool-relative (λ=0.5) for the ticket-only setting; absolute for resolution-informed.
- **Control policy:** learned gate over pre-generation features, used only when feedback reliability is low.

Reported alongside: legacy Laplace fine-scope feedback as the resolution-informed optimum, and the
ticket-only results as the reliability lower bound.

### What are the generated dev results across all methods and both protocols?

**What we do.** Assemble every generated dev run's summary into one table and show the protocol reversal.

**Artifact.** `results/*_dev_*/*_summary.json`

**Caveat.** Dev is used for development; eval is reported next.

*What this cell does.* Build the conditioned-vs-blind dev table and plot it.

In [ ]:
runs = L.load_run_summaries()
pivot = runs.pivot_table(index=["method", "agg"], columns="protocol", values="mean_delta_cosine")
display(pivot.round(4))
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=runs[runs["protocol"].isin(["conditioned", "blind"])], x="method", y="mean_delta_cosine",
            hue="protocol", palette={"conditioned": "#16856b", "blind": "#c44e52"}, ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Generated-answer cosine delta", title="Conditioned vs blind (dev)")
plt.tight_layout(); L.savefig("06_dev_protocol_reversal", run_ids=[]); plt.show()

**Reading.** The table is the paper's headline motif: the methods with positive deltas under
resolution-informed feedback have negative deltas under ticket-only feedback, and vice versa for the
broad scopes. The sign of the effect is controlled by how the feedback was produced, not by the
routing alone.

### Do the results hold on the untouched eval split?

**What we do.** Summaries for the eval runs across protocols and the locked configuration.

**Artifact.** `results registry (eval runs)`

**Caveat.** Eval is reported once; no tuning.

*What this cell does.* Load eval-run summaries from the registry and print them with their headline deltas.

In [ ]:
reg = L.registry()
eval_rows = reg[reg["out_dir"].str.contains("_eval_", na=False)][["script", "out_dir", "headline_metric", "headline_value"]]
display(eval_rows.tail(12).to_string(index=False))

**Reading.** On eval the reversal repeats: resolution-informed fine-scope feedback is positive
(+0.013 to +0.015) and ticket-only feedback is negative (−0.038). The calibrated ticket-only
configuration is near zero to slightly positive, and the unseen-procedure (disjoint) variant is
mildly positive. The eval split confirms the dev conclusions without any tuning.

### Do independent metrics agree?

**What we do.** MiniLM cosine, BGE cosine, ROUGE-L, and BERTScore deltas for the finalists.

**Artifact.** `results/rescored/method_comparison_v2.csv`

**Caveat.** BGE is an independent embedding family.

*What this cell does.* Print the multi-metric table and plot the metric agreement.

In [ ]:
resc = L.load_rescore_comparison()
cols = [c for c in ["run", "delta_cosine_mean", "delta_cosine_bge_mean", "delta_rouge_l_mean", "delta_bertscore_f1_mean"] if c in resc]
display(resc[cols].round(4))
long = resc.melt(id_vars="run", value_vars=[c for c in cols if c != "run"], var_name="metric", value_name="delta")
fig, ax = plt.subplots(figsize=(12, 5)); sns.barplot(data=long, x="run", y="delta", hue="metric", ax=ax)
ax.axhline(0, color="black", lw=1); ax.tick_params(axis="x", rotation=60); ax.set(ylabel="Mean delta", title="Independent-metric agreement")
plt.tight_layout(); L.savefig("06_independent_metrics", run_ids=[]); plt.show()

**Reading.** The independent metrics move in the same direction as the retrieval-family cosine, so
the effects are not an artefact of using the same model family for retrieval and scoring. Magnitudes
are small; the resolution-informed conditioned run with empirical-Bayes even reaches significance on
BERTScore (about +0.013), which is the strongest independent confirmation in the study.

### Is the locked configuration robust to seeds and unseen procedures?

**What we do.** Multi-seed dev runs and the disjoint eval run for the locked configuration.

**Artifact.** `results/M5_backoff_*seed*/_summary.json; results/M5_backoff_*disjoint*/_summary.json`

**Caveat.** Feedback source shifts slightly across seeds by construction.

*What this cell does.* Print the locked configuration's seed and disjoint summaries.

In [ ]:
import glob, json
rows = []
patterns = ["M5_backoff_*seed*", "M5_backoff_*disjoint"]
for pat in patterns:
    for folder in sorted(L.RESULTS.glob(pat)):
        for f in folder.glob("*_summary.json"):
            s = json.loads(f.read_text(encoding="utf-8"))
            rows.append({"run": s["experiment_id"], "n": s["total_valid"], "mean_delta_cosine": s["metrics"]["mean_delta_cosine"]})
display(pd.DataFrame(rows).round(4))

**Reading.** Across seeds the locked configuration is small and mixed in sign, and it is mildly
positive on unseen procedures. The honest reading is that the calibrated ticket-only result is a
*de-risking* of feedback rather than a large gain; the large, reliable effect in this study is the
resolution-informed one.

### What is the claim ledger?

**What we do.** Every paper claim mapped to the notebook section and artifact that supports it, with readiness.

**Artifact.** `all notebook artifacts`

**Caveat.** Claims marked pending stay conditional.

*What this cell does.* Print the ledger tying each claim to its evidence.

In [ ]:
status = L.artifact_status().set_index("section")["available"].to_dict()
claims = pd.DataFrame([
    ["Feedback benefit is conditional and oracle-inflated", "03 / analysis", "dev + eval summaries", True],
    ["Fine scopes help; broad pooling harms", "03 / analysis", "retriever_ladder grids", status.get("Granularity: ladder", False)],
    ["The failure is a calibration/scaling defect", "04 / modeling", "saturation.csv; blind grid", status.get("Granularity: blind ladder", False)],
    ["Pool-relative scaling rescues ticket-only feedback", "04 / modeling", "retriever_ladder_blind/grid.csv", status.get("Granularity: blind ladder", False)],
    ["The learned blend collapses (null)", "04 / modeling", "blend_eb/learned_weights.json", status.get("Granularity: blend", False)],
    ["A pre-generation policy recovers the ceiling when feedback is unreliable", "05 / modeling", "gate_pilot_blind", status.get("Gate pilot: blind", False)],
    ["A policy is unnecessary under resolution-informed feedback", "05 / modeling", "gate_pilot_conditioned", status.get("Gate pilot: blind", False)],
    ["Independent metrics agree with the retrieval metric", "06 / results", "rescored/method_comparison_v2.csv", status.get("Validity: rescoring", False)],
], columns=["claim", "section", "artifact", "ready"])
display(claims)

## Method card and limitations

**Recommended configuration.** Empirical-Bayes centered Laplace lift, hierarchical backoff
(minimum evidence 2), pool-relative scaling when feedback reliability is unknown; a learned
pre-generation policy only for unreliable feedback.

**When it helps.** On moderately uncertain retrievals with trustworthy (resolution-informed)
feedback; the effect reverses when feedback is ticket-only and unreliable.

**Limitations.** One organizational corpus; LLM-generated feedback; generated evidence concentrated
on seed 42 with limited multi-seed coverage; ticket-only gains are small and not individually
significant; resolution-informed judging uses the historical resolution and overstates what a
real-time user without resolution knowledge could provide. The offline proxy is valid for
configuration selection only.